# 05 · Persiapan Data Meteorologi — Bab 6

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 6: memuat data, QC, eksplorasi, *feature engineering*, normalisasi, dan split berbasis waktu. Data contoh sintetik disediakan agar dapat dijalankan tanpa koneksi eksternal; ganti dengan data nyata terbuka (GHCN-Daily/CHIRPS + ERA5/ERA5-Land + indeks iklim) untuk proyek.

## 1. Setup & Data Contoh

Notebook otomatis memakai **data nyata terbuka** bila tersedia (ERA5-Land jakarta + CHIRPS + indeks iklim, via `scripts/download_*.py`); bila file belum ada, dimulai dari sintetik yang meniru perilaku hujan monsun (musiman + hari nol + ekor panjang).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(42)

# ---------------------------------------------------------------------------
# Data contoh: SINTETIK default; bila file data nyata terbuka tersedia
# (ERA5-Land + CHIRPS + indeks iklim dari scripts/download_*.py), pakai itu.
# ---------------------------------------------------------------------------
def _buscar_raiz():
    """Cari pasta repo buku yang memuat manuscripts/ (portabel Colab+local)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "manuscripts").exists():
            return p
    return Path.cwd()

_BASE = _buscar_raiz() / "manuscripts/ch-09-studi-kasus-curah-hujan-terbuka/data"

def cargar_nyata():
    """Ambil data nyata terbuka (ERA5-Land jakarta + CHIRPS) bila ada."""
    era_files = sorted((_BASE / "era5" / "jakarta").glob("era5land_jakarta_*_daily.csv"))
    if not era_files:
        return None
    era = pd.concat([pd.read_csv(p, parse_dates=["tanggal"]).set_index("tanggal")
                     for p in era_files]).sort_index()
    era = era[~era.index.duplicated(keep="first")]
    df = pd.DataFrame({
        "r_hujan": era["tp_mm"],                       # target: hujan grid ERA5-Land (mm/hari)
        "suhu": era["t2m_c"],                          # suhu 2 m (degC)
        "angin": (era["u10"] ** 2 + era["v10"] ** 2) ** 0.5,   # speed angin (m/s)
    })
    return df

df = cargar_nyata()
if df is not None:
    ETIQUETA = "data nyata terbuka: ERA5-Land (jakarta) + indeks iklim nyata"
else:
    ETIQUETA = "data contoh sintetik (ganti dengan data nyata terbuka)"
    t = pd.date_range("2010-01-01", "2021-12-31", freq="D")
    doy = t.dayofyear.values
    musim = 15 + 18 * np.sin(2 * np.pi * (doy - 15) / 365.25) * (np.sin(2*np.pi*(doy-15)/365.25) > 0)
    musim = np.maximum(musim, 0)
    hujan = np.maximum(np.random.gamma(1.2, 6, len(t)) + musim + 3*np.random.randn(len(t)), 0)
    hujan = np.where(np.random.rand(len(t)) < 0.55, 0.0, hujan)
    suhu = 27 + 1.5*np.sin(2*np.pi*(doy-60)/365.25) + 1.0*np.random.randn(len(t))
    kelembapan = 75 + 8*np.sin(2*np.pi*(doy-15)/365.25) + 5*np.random.randn(len(t))
    df = pd.DataFrame({"r_hujan": hujan.round(1), "suhu": suhu.round(1),
                       "kelembapan": kelembapan.round(1)}, index=t)
    df.loc[df.index.day == 29, "r_hujan"] = np.nan  # sisipkan gap contoh (sintetik)

print("Sumber:", ETIQUETA)
print(df.head())
print(df.describe())

Sumber: data nyata terbuka: ERA5-Land (jakarta) + indeks iklim nyata
            r_hujan     suhu     angin
tanggal                               
2018-01-01  15.8692  26.8267  1.985895
2018-01-02   5.9667  26.7146  1.006763
2018-01-03  23.4615  26.7301  0.443239
2018-01-04  39.9425  26.2798  1.523557
2018-01-05  27.4840  25.5465  1.979515
           r_hujan         suhu        angin
count  2922.000000  2922.000000  2922.000000
mean     16.219103    26.342256     0.975952
std      17.504883     0.912148     0.746688
min       0.000400    23.768800     0.015731
25%       3.231450    25.749100     0.418092
50%      11.483700    26.262350     0.758244
75%      23.699425    26.828475     1.340753
max     248.207400    29.997300     4.853539


## 2. Quality Control & Imputasi

In [2]:
print("Nilai hilang per kolom:")
print(df.isna().sum())

# imputasi sederhana: hujan gap -> 0 (konservativ), kolom kontinu -> ffill/mean
df_clean = df.copy()
for c in df_clean.columns:
    if c == "r_hujan":
        df_clean[c] = df_clean[c].fillna(0.0)
    elif c != "chirps_mm":          # kolom verifikasi dipertahakan as-is
        df_clean[c] = df_clean[c].ffill().fillna(df_clean[c].mean())
print("Sisa hilang:", int(df_clean.isna().sum().sum()))

Nilai hilang per kolom:
r_hujan    0
suhu       0
angin      0
dtype: int64
Sisa hilang: 0


## 3. Eksplorasi Distribusi (Gambar 6.1)

In [3]:
plt.figure(figsize=(6.5,4))
df_clean["r_hujan"].hist(bins=60, color="#4a90e2", edgecolor="white")
plt.xlabel("Curah hujan harian (mm)"); plt.ylabel("Frekuensi")
plt.title("Distribusi curah hujan harian (contoh)")
plt.tight_layout(); plt.show()

## 3b. Korelasi Silang (Tujuan Bab 6 #3)

Selang korelasi Pearson antar fitur — petunjuk fitur redundan (untuk Bab 7/9:
fitur yang tinggi berkorelasi duplikasi informasi, tidak selalu menaikkan skiil).

In [ ]:
corr = df_clean.corr(numeric_only=True)
plt.figure(figsize=(6, 4.5))
plt.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=90)
plt.yticks(range(len(corr)), corr.columns)
plt.colorbar(label="Korelasi Pearson")
plt.title("Matriks korelasi silang")
plt.tight_layout(); plt.show()
print(corr.round(2))

### Catatan format NetCDF/GRIB (Tujuan Bab 6 #2)

Notebook ini memakai CSV dedicata. Format grid **NetCDF** (`.nc`) dan **GRIB** dibahas
di teks §6.3: skrip `scripts/download_era5.py` unduh `.nc` (xarray/netCDF4) lalu konversi
ke `*_daily.csv` via `--process` — hasilnya file yang dipakai di sini (bila data nyata
tersedia). Untuk GRIB (mis. reanalisis tekanan/angin grid), panduan loading di teks.

## 4. Feature Engineering

Fitur: deret tunda, musiman sinus, dan (di sini) dummy indeks iklim.

In [4]:
from sklearn.preprocessing import StandardScaler

df_feat = df_clean.copy()
for lag in [1, 2, 3, 7, 14]:
    df_feat[f"hujan_t{lag}"] = df_feat["r_hujan"].shift(lag)
    for v in ("suhu", "angin", "kelembapan", "chirps_mm"):
        if v in df_feat:
            df_feat[f"{v}_t{lag}"] = df_feat[v].shift(lag)

df_feat["mus_sin"] = np.sin(2 * np.pi * df_feat.index.dayofyear / 365.25)
df_feat["mus_cos"] = np.cos(2 * np.pi * df_feat.index.dayofyear / 365.25)

# indeks iklim: RMM harian (BoM) & ONI monthly (CPC) bila tersedia; else dummy
_RAW = _buscar_raiz() / "manuscripts/ch-09-studi-kasus-curah-hujan-terbuka/data/raw"
if (_RAW / "indeks_rmm.csv").exists():
    rmm = pd.read_csv(_RAW / "indeks_rmm.csv", parse_dates=["tanggal"]).set_index("tanggal")
    rmm = rmm[~rmm.index.duplicated(keep="first")]
    df_feat["rmm1"] = rmm["rmm1"].reindex(df_feat.index, method="ffill")
else:
    df_feat["rmm1"] = 0.8 * np.sin(2 * np.pi * np.arange(len(df_feat)) / 45.0)
if (_RAW / "indeks_oni.csv").exists():
    oni = pd.read_csv(_RAW / "indeks_oni.csv", parse_dates=["tanggal"]).set_index("tanggal")
    oni = oni[~oni.index.duplicated(keep="first")]
    df_feat["nino34"] = oni["anom"].reindex(df_feat.index, method="ffill")
else:
    df_feat["nino34"] = 0.5 + 1.2 * np.sin(2 * np.pi * np.arange(len(df_feat)) / 365.25 * 3)

feat_cols = [c for c in df_feat.columns if c != "r_hujan"]
print("Fitur:", feat_cols)

Fitur: ['suhu', 'angin', 'hujan_t1', 'suhu_t1', 'angin_t1', 'hujan_t2', 'suhu_t2', 'angin_t2', 'hujan_t3', 'suhu_t3', 'angin_t3', 'hujan_t7', 'suhu_t7', 'angin_t7', 'hujan_t14', 'suhu_t14', 'angin_t14', 'mus_sin', 'mus_cos', 'rmm1', 'nino34']


## 5. Housekeeping: buang baris awal (lag -> NaN) & split berbasis waktu

In [5]:
df_ml = df_feat.dropna().copy()
data = df_ml[feat_cols]
target = df_ml["r_hujan"]

n = len(data)
n_train = int(n * 0.7); n_val = int(n * 0.15)

scale = StandardScaler().fit(data.iloc[:n_train])
X_train = scale.transform(data.iloc[:n_train])
X_val = scale.transform(data.iloc[n_train:n_train+n_val])
X_test = scale.transform(data.iloc[n_train+n_val:])
y_train, y_val, y_test = target.iloc[:n_train], target.iloc[n_train:n_train+n_val], target.iloc[n_train+n_val:]

print("train", X_train.shape, "| val", X_val.shape, "| test", X_test.shape)
print("Rentang:", df_ml.index[0].date(), "->", df_ml.index[-1].date())

train (2035, 21) | val (436, 21) | test (437, 21)
Rentang: 2018-01-15 -> 2025-12-31


## 6. Latihan Mini

1. Ulangi dengan transformasi `log1p` pada target hujan; bandingkan distribusinya.
2. Buat fungsi pipeline reusable `make_dataset(csv_path) -> X, y, dates, scaler` untuk Bab 8–9.
3. Ganti dummy MJO/ENSO dengan data nyata (panduan tautan di Bab 6) dan periksa efektnya pada model.
4. Terapkan *walk-forward* sederhana dan bandingkan MAE dengan split tunggal.